In [1]:
####################################################
####################################################
## test the motion mapper script converted from Matlab 
## original code from https://github.com/PrincetonUniversity/MouseMotionMapper/tree/main
## converted by AI 
####################################################
####################################################

In [165]:
from __future__ import annotations

import os
import sys
import numpy as np

import matplotlib
import skimage

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from helpers import *

#import mousemotionmapper as mmm

In [166]:
#sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
verbose = True

In [167]:
def make_synthetic_recording(n_frames, n_bodyparts=17, fs=80.0, rng=None):
    """Create one synthetic recording of 2-D body-part trajectories.

    The mouse is modelled as a set of body parts whose relative geometry
    oscillates according to one of several motifs (different frequencies), so
    the distance/wavelet features carry real structure for t-SNE to separate.
    """
    rng = np.random.default_rng() if rng is None else rng
    t = np.arange(n_frames) / fs

    # Base resting layout of the body parts (roughly along the body axis).
    base = np.zeros((n_bodyparts, 2))
    base[:, 0] = np.linspace(-5, 5, n_bodyparts)          # x along the body
    base[:, 1] = 0.5 * rng.standard_normal(n_bodyparts)   # small y offsets

    coords = np.zeros((n_bodyparts, 2, n_frames))

    # Three motifs with distinct oscillation frequencies (Hz).
    motif_freqs = [1.5, 5.0, 12.0]
    seg_len = n_frames // 6
    labels = np.zeros(n_frames, dtype=int)

    pos = 0
    while pos < n_frames:
        motif = rng.integers(0, len(motif_freqs))
        f = motif_freqs[motif]
        end = min(pos + seg_len, n_frames)
        labels[pos:end] = motif
        local_t = t[pos:end]
        # Each body part wiggles with a part-specific phase & amplitude.
        for b in range(n_bodyparts):
            amp = 0.6 + 0.4 * (b / n_bodyparts)
            phase = 2 * np.pi * b / n_bodyparts
            coords[b, 0, pos:end] = base[b, 0] + amp * np.sin(2 * np.pi * f * local_t + phase)
            coords[b, 1, pos:end] = base[b, 1] + amp * np.cos(2 * np.pi * f * local_t + phase)
        pos = end

    # A little tracking noise.
    coords += 0.05 * rng.standard_normal(coords.shape)
    return coords, labels


In [253]:
rng = np.random.default_rng(0)

# ------------------------------------------------------------------ params
params = RunParameters()    

# Keep the synthetic demo fast but representative.
params.pcaModes = 6 # number of PCA modes kept and used downstream, setRunParameters default = 50; run_MMM_pipeline.m uses 10.
params.numProjections = 6 # number of projection columns used to build the wavelets, run_MMM_pipeline.m uses numProjections = 10.
params.numPeriods = 25 # # number of wavelet frequency channels (MATLAB default: 25) 
params.trainingSetSize = 1000          # templates kept per file
params.training_numPoints = 3000    # sub-sample size per file 
params.samplingFreq = 80.0 
params.minF = 0.25
params.maxF = 20.0
params.perplexity = 10.0 
params.kdNeighbors = 50 # local neighborhood size for the template-density sigma (kdNeighbors).
params.tsne_seed = 0

In [254]:
# --------------------------------------------------------------- import SLEAP csv files
import numpy as np

parts = ["nose", "eyeL", "eyeR", "head", "torso", "tail_base", "tail_2", "tail_3", "tail_tip"]  # your skeleton, in the order you want
coord_list, parts = load_sleap_files(
    ["/Volumes/groups/tanaka/People/current/jiwang/projects/image_analysis/Deniz_video_sleap/sleap_test/a2_IR_prediction.csv"], 
    #["rec1.csv", "rec2.csv"],
    bodyparts=parts,     # keeps these, in this order (None = all nodes)
    track=None,          # track name or index if a file has several animals
    min_score=None,       # optional: points below this confidence count as missing
    interpolate=True,    # fills missing points linearly
    max_gap=None)        # optional: don't fill gaps longer than this many frames
bodypart_indices = np.arange(len(parts))

In [255]:
print(parts)
print(len(parts))

['nose', 'eyeL', 'eyeR', 'head', 'torso', 'tail_base', 'tail_2', 'tail_3', 'tail_tip']
9


In [256]:
print(len(coord_list))
print(coord_list[0].shape)

1
(9, 2, 3784)


In [257]:
# --------------------------------------------------------------- synthesise 11 bodypart 
#n_files = 5
#coord_list = []
#label_list = []
#for _ in range(n_files):
#    coords, labels = make_synthetic_recording(4000, n_bodyparts=17, fs=80.0, rng=rng)
#    coord_list.append(coords)
#    label_list.append(labels)
#print(f"Created {n_files} synthetic recordings of shape {coord_list[0].shape} "
#          f"(bodyparts, dims, frames)")

# Use the first 11 body parts for distances (demo subset).
#bodypart_indices = np.arange(11)

In [258]:
bodypart_indices

array([0, 1, 2, 3, 4, 5, 6, 7, 8])

In [259]:
# ----------------------------------------------------------- 
# dissect each step of mmm
# ----------------------------------------------------------- 

In [260]:
# set parameters
rng = np.random.default_rng(0) if rng is None else rng
parameters = params

In [261]:
if verbose:
    print("[1/4] Computing smoothed pairwise distances ...")
distances = coordinates_to_distances(coord_list, bodypart_indices, parameters)

[1/4] Computing smoothed pairwise distances ...


In [262]:
if verbose:
    print("[2/4] Fitting online PCA ...")
pca = fit_online_pca(distances, parameters, rng=rng)

[2/4] Fitting online PCA ...


In [263]:
if verbose:
    print("[3/4] Building training set (wavelets + sub-sample t-SNE) ...")
data_all = build_training_set(distances, pca, parameters, rng=rng, verbose=verbose)

[3/4] Building training set (wavelets + sub-sample t-SNE) ...
  [training set] file 1/1


In [264]:
if verbose:
    print(f"[4/4] Final t-SNE of {data_all.shape[0]} training points ...")
embedding = embed_training_set(data_all, parameters)

[4/4] Final t-SNE of 1000 training points ...


In [265]:
#parameters.perplexity = 50.0 
#parameters.kdNeighbors = 30 # local neighborhood size for the template-density sigma (kdNeighbors).
#parameters.tsne_seed = 0
#parameters.training_perplexity=30.0
#embedding = embed_training_set(data_all, parameters)
print(parameters)

RunParameters(numPeriods=25, omega0=5.0, samplingFreq=80.0, minF=0.25, maxF=20.0, pcaModes=6, numProjections=6, pca_batchSize=20000, trainingSetSize=1000, training_numPoints=3000, kdNeighbors=50, minTemplateLength=1, perplexity=10.0, training_perplexity=20.0, training_relTol=0.002, num_tsne_dim=2, tsne_seed=0, reembed_perplexity=32.0, sigmaTolerance=1e-05, maxNeighbors=200, maxOptimIter=100, embedding_batchSize=20000, smooth_span=5, medfilt_window=5)


In [266]:
print(embedding.shape)

(1000, 2)


In [267]:
# ------------------------------------------------- demo re-embedding step
print("\n--- Re-embedding recording #0 into the training embedding ---")
proj_re = pca.project(distances[0].T, params.numProjections)
    
# Re-embed a small slice so the demo is quick (per-point Nelder-Mead).
z, stats = find_embeddings(proj_re[:300], data_all, embedding, params)
print(f"re-embedded points     : {z.shape}")
print(f"fraction inside hull   : {stats['in_conv_hull'].mean():.2f}")


--- Re-embedding recording #0 into the training embedding ---
re-embedded points     : (300, 2)
fraction inside hull   : 0.99


In [268]:
# ------------------------------------------------------------------ plot
out_png = os.path.join("/Users/jingkui.wang/workspace/imp/image_analysis_test/scripts/my_vame_sleap/motionMapper", "axolotl_embedding.png")
plt.figure(figsize=(6, 6))
plt.scatter(embedding[:, 0], embedding[:, 1], s=10, alpha=0.6, label="training set")
plt.scatter(z[:, 0], z[:, 1], s=10, c="orange", alpha=0.6, label="re-embedded #0")
#plt.scatter(z[:, 0], z[:, 1], s=8,  alpha=0.6)
plt.legend()
plt.title("MotionMapper (Python) t-SNE embedding")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.tight_layout()
plt.savefig(out_png, dpi=120)
print(f"\nSaved embedding scatter to: {out_png}")
print("\nDone — pipeline ran end-to-end successfully.")


Saved embedding scatter to: /Users/jingkui.wang/workspace/imp/image_analysis_test/scripts/my_vame_sleap/motionMapper/axolotl_embedding.png

Done — pipeline ran end-to-end successfully.
